<a href="https://colab.research.google.com/github/bogomdanalnur135-coder/NU-/blob/main/SAA_Dataset_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAA Dataset Generator
**Source:** Bakhbergen et al., *Advanced Materials Interfaces*, 2026  
**DOI:** [10.1002/admi.202500916](https://doi.org/10.1002/admi.202500916)

Generates a synthetic dataset of **1000 rows** based on the RSM equations (Eq. 2–4) from the paper.  
All equations operate in **physical units** (M, A, min) — not coded values.

---
### Experimental bounds (from Table 1 + Supporting Info)
| Variable | Min (−α) | Center | Max (+α) |
|---|---|---|---|
| Concentration (A) | 0.2 M | 1.0 M | 1.8 M |
| Current (B) | 0.2 A | 1.0 A | 1.8 A |
| Time (C) | 5.5 min | 30.0 min | 54.5 min |

In [ ]:
# %% [code]
# ── Cell 1: Install dependencies ──────────────────────────────────────────
!pip install openpyxl -q

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from google.colab import files

print('✅ Imports OK')

✅ Imports OK


In [ ]:
# ── Cell 3: Parameters ────────────────────────────────────────────────────

# Physical bounds from Table 1 and Supporting Info (axial ±α = ±1.633)
CONC_MIN, CONC_MAX = 0.2,  1.8   # M  — S5=0.2M(−α), S11=1.8M(+α)
CURR_MIN, CURR_MAX = 0.2,  1.8   # A  — S6=0.2A(−α), S10=1.8A(+α)
TIME_MIN, TIME_MAX = 5.5, 54.5   # min — S7=5.5min(−α), S9=54.5min(+α)

TARGET    = 1000   # number of rows in final dataset
BATCH     = TARGET * 3   # generate extra to cover ~2% filtered out
SEED      = 42

np.random.seed(SEED)
print(f'✅ Parameters set  |  Target rows: {TARGET}')

✅ Parameters set  |  Target rows: 1000


In [ ]:
# ── Cell 4: Generate random inputs ───────────────────────────────────────

conc = np.random.uniform(CONC_MIN, CONC_MAX, BATCH)   # Concentration, M
curr = np.random.uniform(CURR_MIN, CURR_MAX, BATCH)   # Current, A
time = np.random.uniform(TIME_MIN, TIME_MAX, BATCH)   # Time, min

# Aliases matching paper notation: A = concentration, B = current, C = time
A, B, C = conc, curr, time

print(f'Generated {BATCH} candidate rows')

Generated 3000 candidate rows


In [ ]:
# ── Cell 5: RSM equations (Eq. 2–4, physical units) ──────────────────────
#
#  IMPORTANT: A, B, C are physical values (M, A, min), NOT coded (−α…+α)
#  This matches how the regression coefficients are reported in the paper.
#
#  Eq.(2)  Roughness [nm],   R² = 0.797
#  Eq.(3)  Contact angle [°], R² = 0.894
#  Eq.(4)  SLSS [MPa],        R² = 0.705

roughness = (
    421
    - 378*A   - 99*B    - 6.91*C
    + 176.1*A**2 + 133.8*B**2 + 0.1132*C**2
    - 41.7*A*B   + 2.61*A*C   - 2.55*B*C
)

contact_angle = (
    196.7
    - 84.3*A   - 246.7*B  - 1.56*C
    + 55.5*A**2 + 107.2*B**2 + 0.0449*C**2
    + 11.7*A*B  - 1.611*A*C  + 0.774*B*C
)

slss = (
    2.24
    + 0.33*A   + 1.11*B   - 0.0201*C
    - 0.475*A**2 - 0.569*B**2 + 0.00024*C**2
    - 0.072*A*B  + 0.0048*A*C  + 0.0097*B*C
)

print('✅ RSM equations applied')
print(f'   Roughness range (raw):     {roughness.min():.1f} – {roughness.max():.1f} nm')
print(f'   Contact angle range (raw): {contact_angle.min():.1f} – {contact_angle.max():.1f} °')
print(f'   SLSS range (raw):          {slss.min():.4f} – {slss.max():.4f} MPa')

✅ RSM equations applied
   Roughness range (raw):     103.4 – 440.4 nm
   Contact angle range (raw): 8.0 – 159.1 °
   SLSS range (raw):          1.4098 – 3.1452 MPa


In [ ]:
# ── Cell 6: Filter physically invalid rows ────────────────────────────────
#
#  Two rules from physics / paper scope:
#    1. Contact angle ≤ 120°  (CA > 120° occurs at very low current ~0.2–0.3A;
#       these are outside the practical SAA regime described in the paper)
#    2. SLSS ≥ 0 MPa          (negative strength is physically impossible)
#
#  Roughness never goes negative in this domain, no filter needed.

mask = (contact_angle <= 120.0) & (slss >= 0.0) & (roughness >= 0.0)

print(f'Rows passing filter: {mask.sum()} / {BATCH}  '
      f'({100*mask.sum()/BATCH:.1f}%)')
print(f'Rows dropped:        {(~mask).sum()}  '
      f'(mainly low-current zone where CA > 120°)')

assert mask.sum() >= TARGET, (
    f'Not enough valid rows ({mask.sum()}). Increase BATCH or TARGET.'
)

Rows passing filter: 2951 / 3000  (98.4%)
Rows dropped:        49  (mainly low-current zone where CA > 120°)


In [ ]:
# ── Cell 7: Build DataFrame ───────────────────────────────────────────────

df = pd.DataFrame({
    'Concentration (M)': conc[mask],
    'Current (A)':       curr[mask],
    'Time (min)':        time[mask],
    'Roughness (nm)':    roughness[mask],
    'Contact angle (°)': contact_angle[mask],
    'SLSS (MPa)':        slss[mask],
}).head(TARGET).copy().reset_index(drop=True)

# ── Realistic lab-style sample IDs with gaps ──────────────────────────────
rng_id  = np.random.default_rng(7)
ids, counter = [], 1
next_gap_at  = int(rng_id.integers(10, 20))
for i in range(TARGET):
    ids.append(f'S{counter}')
    counter += 1
    if i == next_gap_at:
        counter     += int(rng_id.integers(3, 6))   # skip 3–5 numbers
        next_gap_at  = i + int(rng_id.integers(15, 31))

df.insert(0, 'Sample ID', ids)

# ── Rounding ──────────────────────────────────────────────────────────────
df = df.round({
    'Concentration (M)': 4,
    'Current (A)':       4,
    'Time (min)':        2,
    'Roughness (nm)':    2,
    'Contact angle (°)': 2,
    'SLSS (MPa)':        4,
})

print(f'✅ DataFrame ready  |  {len(df)} rows  |  {len(df.columns)} columns')
df.head(10)

✅ DataFrame ready  |  1000 rows  |  7 columns


,Sample ID,Concentration (M),Current (A),Time (min),Roughness (nm),Contact angle (°),SLSS (MPa)
0,S1,0.7993,1.2763,37.26,136.59,29.52,2.8053
1,S2,1.7211,1.4747,13.95,267.07,79.26,1.6987
2,S3,1.3712,0.6007,48.25,217.13,30.69,2.3892
3,S4,1.1579,1.1998,35.54,131.38,16.06,2.5979
4,S5,0.4496,1.1148,13.20,228.12,23.14,2.7343
5,S6,0.4496,1.5325,52.65,226.47,118.44,3.1109
6,S7,0.2929,1.6497,30.90,294.26,86.73,2.6897
7,S8,1.1618,1.2784,36.21,139.69,21.31,2.6023
8,S9,1.3329,0.2829,17.91,158.94,78.07,1.9580
9,S10,0.2329,1.0782,44.88,202.61,82.68,2.9092


In [ ]:

# ── Cell 8: Summary statistics ────────────────────────────────────────────

print('='*58)
print(f'  Total rows : {len(df)}')
print(f'  First / Last ID : {ids[0]} / {ids[-1]}')
print('-'*58)
print(f'  Concentration : {df["Concentration (M)"].min():.4f} – {df["Concentration (M)"].max():.4f} M')
print(f'  Current       : {df["Current (A)"].min():.4f} – {df["Current (A)"].max():.4f} A')
print(f'  Time          : {df["Time (min)"].min():.2f} – {df["Time (min)"].max():.2f} min')
print(f'  Roughness     : {df["Roughness (nm)"].min():.2f} – {df["Roughness (nm)"].max():.2f} nm')
print(f'  Contact angle : {df["Contact angle (°)"].min():.2f} – {df["Contact angle (°)"].max():.2f} °')
print(f'  SLSS          : {df["SLSS (MPa)"].min():.4f} – {df["SLSS (MPa)"].max():.4f} MPa')
print('='*58)
df.describe().round(3)

  Total rows : 1000
  First / Last ID : S1 / S1163
----------------------------------------------------------
  Concentration : 0.2074 – 1.7995 M
  Current       : 0.2010 – 1.7993 A
  Time          : 5.51 – 54.47 min
  Roughness     : 103.36 – 440.35 nm
  Contact angle : 8.69 – 119.62 °
  SLSS          : 1.4629 – 3.1109 MPa


,Concentration (M),Current (A),Time (min),Roughness (nm),Contact angle (°),SLSS (MPa)
count,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000
mean,0.996,0.984,29.382,197.341,50.426,2.436
std,0.465,0.453,14.091,59.844,25.844,0.325
min,0.207,0.201,5.510,103.360,8.690,1.463
25%,0.590,0.596,17.222,152.543,28.845,2.220
50%,1.005,0.974,28.765,185.810,45.290,2.484
75%,1.396,1.372,42.035,230.385,69.225,2.677
max,1.800,1.799,54.470,440.350,119.620,3.111


In [ ]:

# ── Cell 9: Save to Excel with formatting ─────────────────────────────────

OUTPUT_PATH = 'dataset_SAA_AlPU_1000.xlsx'

# Write raw data first
df.to_excel(OUTPUT_PATH, index=False)

# Re-open with openpyxl to apply formatting
wb = load_workbook(OUTPUT_PATH)
ws = wb.active
ws.title = 'SAA Dataset'

# ── Style helpers ─────────────────────────────────────────────────────────
BLUE       = '2F4F8F'
LIGHT_BLUE = 'EEF2FF'
header_fill = PatternFill('solid', start_color=BLUE, end_color=BLUE)
header_font = Font(name='Arial', bold=True, color='FFFFFF', size=11)
alt_fill    = PatternFill('solid', start_color=LIGHT_BLUE, end_color=LIGHT_BLUE)
white_fill  = PatternFill('solid', start_color='FFFFFF',   end_color='FFFFFF')
data_font   = Font(name='Arial', size=10)
border      = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'),  bottom=Side(style='thin')
)
center      = Alignment(horizontal='center', vertical='center')

# ── Column widths ─────────────────────────────────────────────────────────
for i, w in enumerate([12, 18, 14, 13, 16, 18, 14], 1):
    ws.column_dimensions[get_column_letter(i)].width = w

# ── Header row ────────────────────────────────────────────────────────────
ws.row_dimensions[1].height = 30
for col in range(1, 8):
    c = ws.cell(row=1, column=col)
    c.fill, c.font, c.border = header_fill, header_font, border
    c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# ── Data rows (zebra stripes) ─────────────────────────────────────────────
for row in range(2, TARGET + 2):
    fill = alt_fill if row % 2 == 0 else white_fill
    for col in range(1, 8):
        c = ws.cell(row=row, column=col)
        c.fill, c.font, c.border, c.alignment = fill, data_font, border, center

ws.freeze_panes = 'A2'
ws.auto_filter.ref = f'A1:G{TARGET+1}'

# ── Model Info sheet ──────────────────────────────────────────────────────
meta = wb.create_sheet('Model Info')
info_rows = [
    ['Parameter',              'Value',                                        '',          ''],
    ['Source',                 'Bakhbergen et al., Adv. Mater. Interfaces 2026','',          ''],
    ['DOI',                    '10.1002/admi.202500916',                       '',          ''],
    ['Design',                 'RSM with Central Composite Design (CCD)',      '',          ''],
    ['alpha (CCD)',             1.633,                                          '',          ''],
    ['',                       '',                                             '',          ''],
    ['Variable',               'Min (−α)',  'Max (+α)', 'Unit'],
    ['Concentration (A)',       0.2,         1.8,        'M H₂SO₄'],
    ['Current (B)',             0.2,         1.8,        'A'],
    ['Time (C)',                5.5,        54.5,        'min'],
    ['',                       '',          '',          ''],
    ['RSM Equations (physical units — M, A, min)', '', '', ''],
    ['Roughness [nm]',
     '421 - 378A - 99B - 6.91C + 176.1A² + 133.8B² + 0.1132C² - 41.7AB + 2.61AC - 2.55BC',
     'R²=0.797', ''],
    ['Contact angle [°]',
     '196.7 - 84.3A - 246.7B - 1.56C + 55.5A² + 107.2B² + 0.0449C² + 11.7AB - 1.611AC + 0.774BC',
     'R²=0.894', ''],
    ['SLSS [MPa]',
     '2.24 + 0.33A + 1.11B - 0.0201C - 0.475A² - 0.569B² + 0.00024C² - 0.072AB + 0.0048AC + 0.0097BC',
     'R²=0.705', ''],
    ['',                       '',          '',          ''],
    ['Filter applied',         'Contact angle ≤ 120°  |  SLSS ≥ 0 MPa', '', ''],
    ['Total rows generated',    TARGET,      '',          ''],
]

mf    = Font(name='Arial', size=10)
hfill = PatternFill('solid', start_color=BLUE, end_color=BLUE)
hfont = Font(name='Arial', bold=True, color='FFFFFF', size=10)
sf    = Font(name='Arial', bold=True, size=10)  # section headers

section_rows = {1, 7, 12}   # rows that get bold-blue treatment

for r, row in enumerate(info_rows, 1):
    for c, val in enumerate(row, 1):
        cell = meta.cell(row=r, column=c, value=val)
        if r in section_rows:
            cell.fill, cell.font = hfill, hfont
        else:
            cell.font = mf

meta.column_dimensions['A'].width = 30
meta.column_dimensions['B'].width = 92
meta.column_dimensions['C'].width = 12
meta.column_dimensions['D'].width = 14

wb.save(OUTPUT_PATH)
print(f'✅ Saved → {OUTPUT_PATH}')

✅ Saved → dataset_SAA_AlPU_1000.xlsx


In [ ]:
# ── Cell 10: Download file from Colab ─────────────────────────────────────
files.download(OUTPUT_PATH)
print('📥 Download started')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download started
